# Video Processing with Gemini Pro 2.5 Flash in BigQuery
This notebook sets up a Google Cloud environment to process video and audio files from a GCS bucket using Vertex AI and BigQuery. It extracts metadata using the Gemini Pro 2.5 Flash model and stores the results in a new BigQuery table.

## 1. Setup Configuration
Fill in these parameters to use a different project or bucket.

In [ ]:
# @title Configuration
PROJECT_ID = "ndy-814-20251212174826" # @param {type:"string"}
GCS_SOURCE_BUCKET = "gs://dashcam_videos" # @param {type:"string"}
DATASET_ID = "video_processing" # @param {type:"string"}
TABLE_ID = "video_details" # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}
CONNECTION_NAME = "vertex-ai-connection" # @param {type:"string"}
MODEL_NAME = "gemini-2.5-flash" # @param {type:"string"}

## 2. Install Libraries and Authenticate
Install the required Python packages and authenticate to Google Cloud.

In [ ]:
!pip install --upgrade google-cloud-bigquery google-cloud-storage vertexai db-dtypes

# If running in Google Colab, uncomment the line below to authenticate:
# from google.colab import auth
# auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
import time
import json
import subprocess

# Configure BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

## 3. Enable APIs
Enable necessary Google Cloud APIs depending on what is needed for BigQuery, Vertex AI, and Cloud Storage.

In [ ]:
!gcloud services enable aiplatform.googleapis.com bigquery.googleapis.com bigqueryconnection.googleapis.com storage.googleapis.com --project {PROJECT_ID}

## 4. Create BigQuery Connection and Assign IAM Roles
BigQuery requires a Cloud Resource connection to interact with Vertex AI remote models and Cloud Storage.

In [ ]:
# Create BQ Connection (Will throw an error if already exists, you can safely ignore it)
!bq mk --connection --location={LOCATION} --project_id={PROJECT_ID} --connection_type=CLOUD_RESOURCE {CONNECTION_NAME} 2>/dev/null || echo "Connection may already exist"

# Get the service account for the created connection
get_sa_cmd = f"bq show --connection --format=json --project_id={PROJECT_ID} --location={LOCATION} {CONNECTION_NAME}"
result = subprocess.run(get_sa_cmd, shell=True, capture_output=True, text=True)

try:
    conn_info = json.loads(result.stdout)
    service_account = conn_info.get('cloudResource', {}).get('serviceAccountId')
    print(f"Connection Service Account: {service_account}")

    if service_account:
        # Assign Vertex AI User role
        !gcloud projects add-iam-policy-binding {PROJECT_ID} \
            --member="serviceAccount:{service_account}" \
            --role="roles/aiplatform.user" > /dev/null

        # Assign Storage Object Admin role to access audio/video files in GCS
        !gcloud projects add-iam-policy-binding {PROJECT_ID} \
            --member="serviceAccount:{service_account}" \
            --role="roles/storage.objectAdmin" > /dev/null
        print("Successfully assigned IAM roles to connection service account.")

        # Wait a moment for IAM bindings to propagate
        time.sleep(15)
except Exception as e:
    print("Error retrieving connection details. Ensure the connection was created successfully.", e)

## 5. Prepare BigQuery Dataset, Object Table, and Remote Model

In [ ]:
# Create Dataset
query = f"CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}` OPTIONS(location='{LOCATION}')"
bq_client.query(query).result()
print("Dataset ready.")

# Create Object Table to access GCS files
object_table_query = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.dashcam_files`
WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['{GCS_SOURCE_BUCKET}/*']
);
"""
bq_client.query(object_table_query).result()
print("Object table ready.")

# Create Remote Model using the specified Gemini Flash version
model_query = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}`
OPTIONS (endpoint = '{MODEL_NAME}');
"""
bq_client.query(model_query).result()
print("Remote model ready.")

## 6. Process Files with Gemini Pro 2.5 Flash
We prompt Gemini to analyze each file in the GCS bucket and enforce JSON formatting to populate our target table.

In [ ]:
# The user wants a summary < 40 words, expanding up to 80 words if danger exists.
prompt = """\
Analyze the contents of this video/audio. Provide the output strictly in valid JSON format. Do not use Markdown backticks. Provide exactly these keys:
{
  \"video_length\": \"duration of the media file\".
  \"number_of_cars_passed\": integer (estimate the count if visible, else 0),
  \"safe_driving_level\": integer from 1 to 10 (1 is safest, 10 is most dangerous. Focus on hazards like reckless driving, unsafe passing, talking on the phone while driving, smoking, lane departure or riding the bumps between lanes.),
  \"drive_description\": \"A summary of the video in less than 40 words. However, if there are dangerous elements, please highlight them and the description can be increased to 80 words.\"
}
"""

extract_query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS
SELECT
  uri AS video_file_name,
  JSON_VALUE(ml_generate_text_llm_result, '$.video_length') AS video_length,
  CAST(JSON_VALUE(ml_generate_text_llm_result, '$.number_of_cars_passed') AS INT64) AS number_of_cars_passed,
  CAST(JSON_VALUE(ml_generate_text_llm_result, '$.safe_driving_level') AS INT64) AS safe_driving_level,
  JSON_VALUE(ml_generate_text_llm_result, '$.drive_description') AS drive_description
FROM
  ML.GENERATE_TEXT(
    MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_model`,
    TABLE `{PROJECT_ID}.{DATASET_ID}.dashcam_files`,
    STRUCT(
      '''{prompt}''' AS prompt,
      TRUE AS flatten_json_output,
      0.2 AS temperature,
      1024 AS max_output_tokens
    )
  )
"""

print(f"Starting processing from {GCS_SOURCE_BUCKET}. This may take some time...")
job = bq_client.query(extract_query)
job.result()
print(f"Successfully processed files and created table {PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")

## 7. Preview the Extracted Data

In [ ]:
preview_query = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` LIMIT 10"
results = bq_client.query(preview_query).to_dataframe()
display(results)